# CSE5280 — Robot-Arm Evacuation Interference Simulation

**Author:** Joshua Cajuste  
**Course:** CSE5280  
**Instructor:** Eraldo Ribeiro  
**Department of Electrical Engineering and Computer Science, Florida Institute of Technology**

---

## Overview

This notebook extends the three-floor building evacuation simulation with an autonomous **robot arm whose goal is to interfere with the evacuation flow**. The robot:

1. **Perceives** the crowd — identifies agents near exits using a proximity filter
2. **Clusters** those agents with k-means to find active evacuation streams
3. **Predicts** where the dominant cluster will be 15 steps in the future using a velocity-extrapolation model
4. **Controls** its end-effector toward the predicted location via Jacobian-based inverse kinematics
5. **Interferes** — the end-effector is treated as a dynamic obstacle; agents repel from it just like they repel each other

### Building Layout
The three floors are arranged **diagonally** (offset in X and Y as well as Z) so the stacked-floor view is an isometric cascade. The robot arm is placed **outside the building near exit 1** (lower-left corner) and its second robot near **exit 2** (upper-right corner).

### New Cost Terms
| Term | Formula | Role |
|------|---------|------|
| $C_{\text{robot}}$ | $s_{\text{bot}} \cdot \max(0,\, r_{\text{bot}} - \|p - e_{\text{eff}}\|)^2$ | Repels agents from the robot end-effector |

All other cost terms remain identical to the previous assignment.

## Table of Contents
1. [Installation & Imports](#1)
2. [Simulation Parameters](#2)
3. [Building Geometry — Diagonal Layout](#3)
4. [Geometry & Surface Helpers](#4)
5. [Cost Functions](#5)
6. [Gradient Computation](#6)
7. [Robot Arm — IK Framework](#7)
8. [Perception — Cluster Detection & Prediction](#8)
9. [Agent Initialization](#9)
10. [3D Scene Construction](#10)
11. [Simulation Loop & Video Export](#11)
12. [Summary & Analysis](#12)

## 1. Installation & Imports <a id='1'></a>

In [ ]:
import sys
!{sys.executable} -m pip install -q numpy vedo imageio[ffmpeg] scikit-learn

In [ ]:
import os
import numpy as np
import imageio
from collections import deque
from sklearn.cluster import KMeans
from vedo import (
    Plotter, Plane, Sphere, Mesh, Arrow, Line, Text2D, settings
)
from IPython.display import Video, display

settings.default_backend = "vtk"
print("Imports OK")


## 2. Simulation Parameters <a id='2'></a>

| Parameter | Value | Description |
|-----------|-------|-------------|
| `W` | 30 | Room width & depth |
| `num_agents` | 30 | Crowd size |
| `alpha` | 0.05 | Gradient descent step size |
| `tau` | 8.0 | Goal softmin temperature |
| `repulsion_radius` | 1.5 | Agent–agent repulsion radius |
| `robot_repulsion_radius` | 3.0 | End-effector obstacle radius |
| `robot_repulsion_strength` | 10.0 | End-effector obstacle strength |
| `PREDICT_HORIZON` | 12 | Steps ahead to predict cluster |
| `steps` | 600 | Total simulation iterations |


In [ ]:
# ── Environment ───────────────────────────────────────────────────────────────
W   = 30.0        # main floor width & depth
W2  = 20.0        # second floor width & depth
FLOOR2_OFFSET_X = W + 14.0   # second floor starts this far right of main floor
FLOOR2_OFFSET_Y = 5.0        # slight Y offset for visual separation
RAMP_WIDTH  = 3.5            # ramp corridor half-width
Z0  = 0.0         # main floor z
Z2  = 0.0         # second floor z (same level — side-by-side)

# ── Crowd ──────────────────────────────────────────────────────────────────────
num_agents         = 35
alpha              = 0.045
tau                = 9.0
ws                 = 0.08
repulsion_radius   = 1.4
repulsion_strength = 3.5
max_step           = 0.55
exit_tol           = 1.3
steps              = 900
TRAIL_LENGTH       = 8

# ── Robot obstacle ─────────────────────────────────────────────────────────────
robot_repulsion_radius   = 3.5
robot_repulsion_strength = 9.0

# ── Wall obstacle ──────────────────────────────────────────────────────────────
wall_band    = 1.0
wall_strength = 6.0

# ── Perception ────────────────────────────────────────────────────────────────
EXIT_DETECT_RADIUS = 10.0
K_CLUSTERS        = 2
PREDICT_HORIZON   = 12
VELOCITY_MEMORY   = 8

# ── Video ─────────────────────────────────────────────────────────────────────
RENDER_EVERY = 4     # render every N steps (higher = slower but smoother video)
VIDEO_FPS    = 12    # lower fps = slower playback

print("Parameters set.")


## 3. Room Geometry <a id='3'></a>

Single-floor square room (30×30 units). Exits are at opposite corners. Three interior pillars
create flow obstacles. Robot arms are placed just outside the two exit corners and reach inside
to intercept evacuating agents.


In [ ]:
import numpy as np

# ── Main floor exits ──────────────────────────────────────────────────────────
# Exit 1: corner of main floor
# Exit 2: on the ramp side wall — agents pass through into floor 2
exit1 = np.array([1.5,  1.5,  Z0])
exit2 = np.array([W-1.5, W-1.5, Z0])   # far corner of main floor

# ── Second floor — placed to the right, connected by a ramp ──────────────────
# Ramp connects right wall of main floor (x=W) to left wall of floor 2
# Ramp spans Y: [W*0.3, W*0.7] at a mid-corridor Y
RAMP_Y_CENTER = W * 0.5
RAMP_XA = W                            # ramp start X (main floor right wall)
RAMP_XB = FLOOR2_OFFSET_X             # ramp end X (floor 2 left wall)
RAMP_YA = RAMP_Y_CENTER               # ramp start Y (in main floor coords)
RAMP_YB = FLOOR2_OFFSET_Y + W2 * 0.5 # ramp end Y (in floor 2 coords)

# Third exit: inside floor 2 (agents that cross the ramp evacuate here)
exit3 = np.array([FLOOR2_OFFSET_X + W2 - 1.5, FLOOR2_OFFSET_Y + 1.5, Z2])
exits = [exit1, exit2, exit3]

# ── Walls ─────────────────────────────────────────────────────────────────────
def room_boundary(x0, y0, x1, y1, gap_side=None, gap_y0=None, gap_y1=None):
    """Axis-aligned room boundary with optional gap on the right wall."""
    segs = []
    # Bottom
    segs.append(((x0,y0),(x1,y0)))
    # Top
    segs.append(((x0,y1),(x1,y1)))
    # Left
    segs.append(((x0,y0),(x0,y1)))
    # Right wall — with gap if requested
    if gap_side == 'right' and gap_y0 is not None:
        segs.append(((x1,y0),(x1,gap_y0)))
        segs.append(((x1,gap_y1),(x1,y1)))
    elif gap_side == 'left' and gap_y0 is not None:
        segs.append(((x0,y0),(x0,gap_y0)))
        segs.append(((x0,gap_y1),(x0,y1)))
    else:
        segs.append(((x1,y0),(x1,y1)))
    return segs

gap_y0 = RAMP_Y_CENTER - RAMP_WIDTH
gap_y1 = RAMP_Y_CENTER + RAMP_WIDTH

walls  = room_boundary(0, 0, W, W,
                        gap_side='right', gap_y0=gap_y0, gap_y1=gap_y1)
walls += room_boundary(FLOOR2_OFFSET_X, FLOOR2_OFFSET_Y,
                        FLOOR2_OFFSET_X+W2, FLOOR2_OFFSET_Y+W2,
                        gap_side='left',
                        gap_y0=RAMP_YB - RAMP_WIDTH,
                        gap_y1=RAMP_YB + RAMP_WIDTH)
# Ramp side walls (the corridor connecting the two rooms)
walls.append(((RAMP_XA, gap_y0),  (RAMP_XB, RAMP_YB - RAMP_WIDTH)))
walls.append(((RAMP_XA, gap_y1),  (RAMP_XB, RAMP_YB + RAMP_WIDTH)))

# Interior pillars in main floor
pillars = [
    (W*0.35, W*0.45, 1.1, 1.1),
    (W*0.65, W*0.45, 1.1, 1.1),
    (W*0.50, W*0.65, 1.1, 1.1),
]

print(f"Exits: {len(exits)}  Walls: {len(walls)}  Pillars: {len(pillars)}")
print(f"Ramp: X [{RAMP_XA:.1f} -> {RAMP_XB:.1f}]  Y [{RAMP_YA:.1f} -> {RAMP_YB:.1f}]")


## 4. Geometry & Surface Helpers <a id='4'></a>

In [ ]:
def point_segment_distance(P, A, B):
    v     = B - A
    denom = np.dot(v, v)
    if denom < 1e-12:
        return np.linalg.norm(P - A)
    t = np.clip(np.dot(P-A, v) / denom, 0, 1)
    return np.linalg.norm(P - (A + t*v))

def pillar_distance(p):
    best = np.inf
    px, py = p[0], p[1]
    for (cx, cy, hw, hd) in pillars:
        dx = max(abs(px-cx) - hw, 0)
        dy = max(abs(py-cy) - hd, 0)
        best = min(best, np.sqrt(dx*dx + dy*dy))
    return best

def in_ramp_corridor(p):
    """True if XY position is inside the ramp corridor between the two floors."""
    x, y = p[0], p[1]
    if x < RAMP_XA or x > RAMP_XB:
        return False
    # Linearly interpolate corridor Y-center and half-width across X
    t  = (x - RAMP_XA) / max(RAMP_XB - RAMP_XA, 1e-9)
    yc = RAMP_YA + t * (RAMP_YB - RAMP_YA)
    return abs(y - yc) < RAMP_WIDTH

print("Helpers defined.")


## 5. Cost Functions <a id='5'></a>

$$C(p) = C_{\text{goal}} + C_{\text{walls}} + C_{\text{smooth}} + C_{\text{repulsion}} + C_{\text{robot}}$$

| Term | Formula | Role |
|------|---------|------|
| $C_{\text{goal}}$ | $-\tau \ln \sum_k e^{-\|p-e_k\|^2/\tau}$ | Attract agent toward nearest exit |
| $C_{\text{walls}}$ | $s_w \max(0, r_w - d_w)^2$ | Repel from outer walls & pillars |
| $C_{\text{smooth}}$ | $w_s \|p - p_{\text{prev}}\|^2$ | Penalise abrupt direction changes |
| $C_{\text{repulsion}}$ | $s_r \max(0, r_r - d_{ij})^2$ | Agent–agent personal space |
| $C_{\text{robot}}$ | $s_{\text{bot}} \max(0, r_{\text{bot}} - d_{\text{eff}})^2$ | Dynamic end-effector obstacle |


In [ ]:
end_effector_positions = [
    np.array([-4.0, 1.5,  0.0]),
    np.array([W+4,  W-1.5, 0.0]),
]

def goal_cost(p):
    costs = np.array([np.linalg.norm(p[:2] - e[:2])**2 for e in exits])
    return -tau * np.log(np.sum(np.exp(-costs / tau)) + 1e-30)

def wall_cost(p):
    cost = 0.0
    for w in walls:
        A = np.array(w[0], dtype=float)
        B = np.array(w[1], dtype=float)
        d = point_segment_distance(p[:2], A, B)
        if d < wall_band:
            cost += wall_strength * (wall_band - d)**2
    dp = pillar_distance(p)
    if dp < wall_band:
        cost += wall_strength * (wall_band - dp)**2
    return cost

def smoothness_cost(p, p_prev):
    return ws * np.linalg.norm(p - p_prev)**2

def repulsion_cost(p, agents):
    cost = 0.0
    for a in agents:
        d = np.linalg.norm(p[:2] - a[:2])
        if 0 < d < repulsion_radius:
            cost += repulsion_strength * (repulsion_radius - d)**2
    return cost

def robot_obstacle_cost(p):
    cost = 0.0
    for eff in end_effector_positions:
        d = np.linalg.norm(p - eff)
        if d < robot_repulsion_radius:
            cost += robot_repulsion_strength * (robot_repulsion_radius - d)**2
    return cost

def total_cost(p, p_prev, agents):
    return (
        goal_cost(p)
        + wall_cost(p)
        + smoothness_cost(p, p_prev)
        + repulsion_cost(p, agents)
        + robot_obstacle_cost(p)
    )

print("Cost functions defined.")


## 6. Gradient Computation <a id='6'></a>

In [ ]:
def gradient(p, p_prev, agents, eps=1e-3):
    g = np.zeros(3)
    for i in range(2):
        dp = np.zeros(3); dp[i] = eps
        g[i] = (total_cost(p+dp, p_prev, agents)
              - total_cost(p-dp, p_prev, agents)) / (2*eps)
    return g

print("Gradient defined.")


## 7. Robot Arm — IK Framework <a id='7'></a>

The `RobotArm` class implements:
- **Forward kinematics**: computes world-frame joint positions and end-effector from joint angles $\Phi$
- **Jacobian IK**: one Newton step per simulation step, so the arm moves smoothly rather than teleporting

Two robot instances are created — one near each exit. They share the same kinematic structure but have different base positions.

In [ ]:
class RobotArm:
    """
    4-DOF robot arm — exact FK architecture from robot_arms_inverse_kinematics.ipynb.
      phi1: base yaw (Z-axis)
      phi2: shoulder pitch (Y-axis)
      phi3: elbow pitch (Y-axis)
      phi4: wrist pitch (Y-axis, kept 0)
    Uses homogeneous transform chain via getLocalFrameMatrix.
    IK: numerical Jacobian pseudo-inverse with target_lambda damping.
    """

    def __init__(self, arm_location, L1=5.5, L2=5.0, L3=4.0, L4=0.0):
        # Store as plain 1D array — avoids all reshape confusion downstream
        self.arm_location   = np.array(arm_location, dtype=float).ravel()  # shape (3,)
        self.L1, self.L2, self.L3, self.L4 = L1, L2, L3, L4
        self.phi            = np.array([30.0, -50.0, -40.0, 0.0])  # degrees
        self.target         = self.arm_location.copy()
        self.target_lambda  = 0.003   # IK step damping (matches original)
        self.delta_phi      = 0.1     # finite-diff step in degrees

    # ── Single-axis rotation matrix (original implementation) ─────────────
    def RotationMatrix(self, theta, axis_name):
        c = np.cos(np.radians(theta))
        s = np.sin(np.radians(theta))
        if axis_name == "x":
            return np.array([[1,0,0],[0,c,-s],[0,s,c]], dtype=float)
        elif axis_name == "y":
            return np.array([[c,0,s],[0,1,0],[-s,0,c]], dtype=float)
        elif axis_name == "z":
            return np.array([[c,-s,0],[s,c,0],[0,0,1]], dtype=float)
        raise ValueError(f"Unknown axis: {axis_name}")

    # ── Homogeneous transform (original implementation) ───────────────────
    def getLocalFrameMatrix(self, R_ij, t_ij):
        """Build 4x4 homogeneous matrix from 3x3 rotation and 3-element translation."""
        t = np.array(t_ij, dtype=float).ravel()   # guarantee 1D shape-(3,)
        return np.block([
            [R_ij,                   t.reshape(3, 1)],
            [np.zeros((1, 3)),       np.ones((1, 1)) ],
        ])

    # ── Forward kinematics (original chain) ───────────────────────────────
    def forward_kinematics(self, Phi=None):
        """Returns (T_00, T_01, T_02, T_03, T_04, e) — all in world frame."""
        if Phi is None:
            Phi = self.phi
        phi1, phi2, phi3, phi4 = float(Phi[0]), float(Phi[1]), float(Phi[2]), float(Phi[3])
        loc = self.arm_location   # plain (3,) array

        # Frame 0 — fixed base plate at (x, y, 0)
        R_00 = self.RotationMatrix(0, "z")
        t_00 = np.array([loc[0], loc[1], 0.0])
        T_00 = self.getLocalFrameMatrix(R_00, t_00)

        # Frame 1 — base yaw around Z
        R_01 = self.RotationMatrix(phi1, "z")
        t_01 = loc.copy()
        T_01 = self.getLocalFrameMatrix(R_01, t_01)

        # Frame 2 — shoulder pitch around Y  (offset L1 up from Frame 1)
        R_12 = self.RotationMatrix(phi2, "y")
        t_12 = np.array([0.0, 0.0, self.L1])
        T_12 = self.getLocalFrameMatrix(R_12, t_12)
        T_02 = T_01 @ T_12

        # Frame 3 — elbow pitch around Y  (offset L2 along segment)
        R_23 = self.RotationMatrix(phi3, "y")
        t_23 = np.array([0.0, 0.0, self.L2])
        T_23 = self.getLocalFrameMatrix(R_23, t_23)
        T_03 = T_02 @ T_23

        # Frame 4 — wrist pitch around Y  (offset L3 along segment)
        R_34 = self.RotationMatrix(phi4, "y")
        t_34 = np.array([0.0, 0.0, self.L3])
        T_34 = self.getLocalFrameMatrix(R_34, t_34)
        T_04 = T_03 @ T_34

        e = T_04[0:3, 3]   # end-effector world position
        return T_00, T_01, T_02, T_03, T_04, e

    def end_effector(self, Phi=None):
        return self.forward_kinematics(Phi)[5]

    def joint_positions(self, Phi=None):
        """World positions: (base_plate, yaw_joint, shoulder, elbow, end_effector)."""
        T_00, T_01, T_02, T_03, T_04, e = self.forward_kinematics(Phi)
        return (T_00[0:3,3], T_01[0:3,3], T_02[0:3,3], T_03[0:3,3], e)

    # ── Numerical Jacobian (original finite-difference) ───────────────────
    def jacobian_matrix(self, Phi):
        """3x3 Jacobian: columns are d(eff)/d(phi_i) for i=0,1,2."""
        step = self.delta_phi
        e0   = self.end_effector(Phi)
        cols = []
        for i in range(3):
            dp = np.zeros(4); dp[i] = step
            cols.append((self.end_effector(Phi + dp) - e0) / step)
        return np.column_stack(cols)   # shape (3, 3)

    # ── One IK step per simulation frame ──────────────────────────────────
    def step_toward_target(self, target):
        """Moves end-effector one step toward target. Returns new end-effector pos."""
        self.target = np.array(target, dtype=float).ravel()
        e    = self.end_effector()
        err  = self.target - e
        J    = self.jacobian_matrix(self.phi)
        # Pseudo-inverse with target_lambda damping (original formulation)
        dphi = np.linalg.pinv(J) @ (self.target_lambda * err)
        self.phi[:3] += np.degrees(dphi)
        # Joint angle limits
        self.phi[0] = np.clip(self.phi[0], -180, 180)
        self.phi[1] = np.clip(self.phi[1],  -90,  20)
        self.phi[2] = np.clip(self.phi[2], -130,  10)
        return self.end_effector()


print("RobotArm (original FK architecture) defined.")


## 8. Perception — Cluster Detection & Prediction <a id='8'></a>

### Cluster Detection
At each step we filter agents that are:
- **Active** (not yet evacuated)
- **Near an exit** — within `EXIT_DETECT_RADIUS` of any exit
- **On the ground floor** (z < H/2)

We then run **k-means** (k=2) on those positions in world XY and pick the cluster with the **most members** as the primary interception target.

### Velocity Prediction
We maintain a rolling queue of the dominant cluster's centroid over the last `VELOCITY_MEMORY` steps. The velocity is estimated as the mean displacement, then extrapolated `PREDICT_HORIZON` steps forward:

$$p_{\text{pred}} = c_{\text{now}} + T \cdot \hat{v}$$

where $T = $ `PREDICT_HORIZON` and $\hat{v}$ is the smoothed velocity.

In [ ]:
from collections import deque
cluster_history = [deque(maxlen=VELOCITY_MEMORY) for _ in range(3)]

def agents_near_exit(agents, active, exit_pos, radius=EXIT_DETECT_RADIUS):
    return [i for i, (p, a) in enumerate(zip(agents, active))
            if a and np.linalg.norm(p[:2] - exit_pos[:2]) < radius]

def detect_primary_cluster(positions_2d):
    if len(positions_2d) < 2:
        return None
    k  = min(K_CLUSTERS, len(positions_2d))
    km = KMeans(n_clusters=k, n_init=3, random_state=0)
    labels = km.fit_predict(positions_2d)
    primary = np.argmax(np.bincount(labels))
    return km.cluster_centers_[primary]

def predict_cluster_position(history_deque, centroid_now):
    history_deque.append(centroid_now.copy())
    if len(history_deque) < 2:
        return centroid_now
    pts      = np.array(history_deque)
    velocity = np.diff(pts, axis=0).mean(axis=0)
    return centroid_now + PREDICT_HORIZON * velocity

def update_robot_targets(agents, active, robots):
    predicted_targets = []
    cluster_centroids = []
    for robot_idx, (robot, exit_pos, hist) in enumerate(
            zip(robots, exits, cluster_history)):
        near_idx = agents_near_exit(agents, active, exit_pos)
        if len(near_idx) < 2:
            eff = robot.end_effector()
            if robot_idx < len(end_effector_positions):
                end_effector_positions[robot_idx] = eff
            predicted_targets.append(None)
            cluster_centroids.append(None)
            continue
        positions_2d = np.array([agents[i][:2] for i in near_idx])
        centroid_2d  = detect_primary_cluster(positions_2d)
        if centroid_2d is None:
            predicted_targets.append(None)
            cluster_centroids.append(None)
            continue
        cluster_centroids.append(centroid_2d)
        predicted_2d = predict_cluster_position(hist, centroid_2d)
        predicted_targets.append(predicted_2d)
        target_3d = np.array([predicted_2d[0], predicted_2d[1], 1.5])
        eff = robot.step_toward_target(target_3d)
        if robot_idx < len(end_effector_positions):
            end_effector_positions[robot_idx] = eff
    return predicted_targets, cluster_centroids

print("Perception defined.")


## 9. Agent Initialization <a id='9'></a>

In [ ]:
np.random.seed(42)

agents = []
# 2/3 of agents on main floor, 1/3 on second floor
n_main  = int(num_agents * 0.65)
n_floor2 = num_agents - n_main

for _ in range(n_main):
    x = np.random.uniform(W*0.1, W*0.85)
    y = np.random.uniform(W*0.1, W*0.85)
    agents.append(np.array([x, y, Z0]))

for _ in range(n_floor2):
    x = np.random.uniform(FLOOR2_OFFSET_X + 1.5, FLOOR2_OFFSET_X + W2 - 1.5)
    y = np.random.uniform(FLOOR2_OFFSET_Y + 1.5, FLOOR2_OFFSET_Y + W2 - 1.5)
    agents.append(np.array([x, y, Z2]))

agents      = np.array(agents)
prev_agents = agents.copy()
active      = np.ones(num_agents, dtype=bool)
trails      = [deque(maxlen=TRAIL_LENGTH) for _ in range(num_agents)]

# ── Robot arms — bases outside the two main-floor exits ──────────────────────
# Robot 0: near exit1 (lower-left of main floor)
# Robot 1: near exit2 (upper-right of main floor)
# Robot 2: near exit3 (inside floor 2, far corner) — guards that floor
robot0 = RobotArm(arm_location=[-5.5,  1.5,  0.0])
robot1 = RobotArm(arm_location=[W+5.5, W-1.5, 0.0])
robot2 = RobotArm(arm_location=[FLOOR2_OFFSET_X + W2 + 5.0,
                                  FLOOR2_OFFSET_Y + 1.5, 0.0])
robots = [robot0, robot1, robot2]

# End-effector positions (mutable, updated each step)
end_effector_positions = [r.end_effector() for r in robots]

print(f"Agents: {num_agents} ({n_main} main / {n_floor2} floor-2)")
print(f"Robots: {len(robots)}")


## 10. 3D Scene Construction <a id='10'></a>

- Dark floor plane with grid lines for depth
- Solid outer walls with realistic thickness
- Three interior pillar obstacles
- Glowing exit portals at opposite corners
- Robot arms rendered as arrow-chain segments with joint spheres
- Cyan wireframe bubble showing end-effector repulsion radius
- Yellow cluster centroid + cyan predicted-position markers appear at runtime


In [ ]:
all_objects = []

# ── Main floor ────────────────────────────────────────────────────────────────
all_objects.append(
    Plane(pos=(W/2, W/2, Z0-0.05), normal=(0,0,1), s=(W, W))
    .color("#0e1c2e").alpha(1.0)
)

# ── Second floor ──────────────────────────────────────────────────────────────
cx2 = FLOOR2_OFFSET_X + W2/2
cy2 = FLOOR2_OFFSET_Y + W2/2
all_objects.append(
    Plane(pos=(cx2, cy2, Z2-0.05), normal=(0,0,1), s=(W2, W2))
    .color("#1a2e1a").alpha(1.0)
)

# ── Ramp floor surface (filled quad) ──────────────────────────────────────────
ramp_pts = [
    [RAMP_XA, gap_y0,               Z0+0.01],
    [RAMP_XA, gap_y1,               Z0+0.01],
    [RAMP_XB, RAMP_YB+RAMP_WIDTH,   Z2+0.01],
    [RAMP_XB, RAMP_YB-RAMP_WIDTH,   Z2+0.01],
]
all_objects.append(
    Mesh([ramp_pts, [[0,1,2,3]]]).color("#2a3a1a").alpha(0.95)
)
# Ramp edge lines
all_objects.append(Line(ramp_pts[0], ramp_pts[3], lw=2).color("#aaffaa").alpha(0.5))
all_objects.append(Line(ramp_pts[1], ramp_pts[2], lw=2).color("#aaffaa").alpha(0.5))

# ── Grid lines ────────────────────────────────────────────────────────────────
def draw_grid(x0, y0, x1, y1, step=5.0, z=0.02, col="#1a3050", alp=0.5):
    objs = []
    for gx in np.arange(x0, x1+step, step):
        objs.append(Line([gx,y0,z],[gx,y1,z], lw=1).color(col).alpha(alp))
    for gy in np.arange(y0, y1+step, step):
        objs.append(Line([x0,gy,z],[x1,gy,z], lw=1).color(col).alpha(alp))
    return objs

all_objects += draw_grid(0, 0, W, W)
all_objects += draw_grid(FLOOR2_OFFSET_X, FLOOR2_OFFSET_Y,
                          FLOOR2_OFFSET_X+W2, FLOOR2_OFFSET_Y+W2,
                          col="#1a3a1a")

# ── Outer walls ────────────────────────────────────────────────────────────────
wall_h = 4.0
def wall_slab(x0, y0, x1, y1, z, h, col="#2a4060", alp=0.65):
    dx, dy = x1-x0, y1-y0
    L = np.sqrt(dx*dx + dy*dy)
    if L < 1e-6:
        return None
    pts = [[x0,y0,z],[x1,y1,z],[x1,y1,z+h],[x0,y0,z+h]]
    return Mesh([pts, [[0,1,2,3]]]).color(col).alpha(alp)

for w in walls:
    s = wall_slab(w[0][0], w[0][1], w[1][0], w[1][1], Z0, wall_h)
    if s:
        all_objects.append(s)

# Second floor walls (green-tint)
for w in walls:
    # Already included above — walls list covers both rooms
    pass

# ── Interior pillars ──────────────────────────────────────────────────────────
pillar_h = 3.5
for (cx, cy, hw, hd) in pillars:
    pts_b = [[cx-hw,cy-hd,Z0],[cx+hw,cy-hd,Z0],[cx+hw,cy+hd,Z0],[cx-hw,cy+hd,Z0]]
    pts_t = [[p[0],p[1],Z0+pillar_h] for p in pts_b]
    for f in [
        [pts_b[0],pts_b[1],pts_t[1],pts_t[0]],
        [pts_b[1],pts_b[2],pts_t[2],pts_t[1]],
        [pts_b[2],pts_b[3],pts_t[3],pts_t[2]],
        [pts_b[3],pts_b[0],pts_t[0],pts_t[3]],
    ]:
        all_objects.append(Mesh([f, [[0,1,2,3]]]).color("#1e3a5a").alpha(0.9))
    all_objects.append(Mesh([pts_t+[pts_t[0]], [[0,1,2,3]]]).color("#2a4a70").alpha(0.9))

# ── Exit portals ──────────────────────────────────────────────────────────────
exit_colors = ["#00ff55", "#ffaa00", "#00aaff"]
exit_spheres = []
for ex, ec in zip(exits, exit_colors):
    all_objects.append(Plane(pos=ex+[0,0,0.05], normal=(0,0,1), s=(3.2,3.2)).color(ec).alpha(0.12))
    all_objects.append(Sphere(pos=ex+[0,0,0.3], r=1.0).color(ec).alpha(0.2).wireframe(True))
    s = Sphere(pos=ex+[0,0,0.3], r=0.6).color(ec).alpha(1.0)
    exit_spheres.append(s)
    all_objects.append(s)

# ── Agent spheres ──────────────────────────────────────────────────────────────
agent_spheres = [
    Sphere(a+[0,0,0.4], r=0.42).color("#ff3333").alpha(1.0) for a in agents
]
all_objects += agent_spheres

# ── Robot arm visuals ─────────────────────────────────────────────────────────
robot_actors = [[] for _ in range(3)]

def make_robot_visuals(robot):
    """Render the 4-DOF arm as Arrow segments using the FK joint positions."""
    positions = robot.joint_positions()   # (base_plate, j1, j2, j3, eff)
    actors = []
    seg_colors = ["#2255cc", "#3a7fff", "#55aaff", "#88ccff"]
    seg_radii  = [0.20,      0.17,      0.14,      0.11]
    for k in range(4):
        a, b = positions[k], positions[k+1]
        if np.linalg.norm(b-a) < 1e-3:
            continue
        actors.append(Arrow(start_pt=a, end_pt=b,
                            shaft_radius=seg_radii[k],
                            head_radius=seg_radii[k]*1.6,
                            head_length=0.01, res=12,
                            c=seg_colors[k], alpha=1.0))
    # Joint spheres
    joint_r   = [0.55, 0.45, 0.40, 0.35]
    joint_col = ["#1a44bb", "#2255cc", "#3366dd", "#4477ee"]
    for k, jpt in enumerate(positions[:4]):
        actors.append(Sphere(pos=jpt, r=joint_r[k]).color(joint_col[k]).alpha(1.0))
    # End-effector glow
    eff = positions[4]
    actors.append(Sphere(pos=eff, r=0.60).color("#ffe000").alpha(1.0))
    actors.append(Sphere(pos=eff, r=0.85).color("#ffaa00").alpha(0.15))
    actors.append(Sphere(pos=eff, r=robot_repulsion_radius).alpha(0.06).wireframe(True).color("#00eeff"))
    return actors

# ── Cluster & prediction markers ──────────────────────────────────────────────
cluster_markers = [Sphere(r=0.55).color("#ffee00").alpha(0.0) for _ in robots]
predict_markers = [Sphere(r=0.55).color("#00ffee").alpha(0.0) for _ in robots]

# ── HUD ──────────────────────────────────────────────────────────────────────
hud = Text2D("Step: 0 | Evacuated: 0/35 (0%)", pos="top-left", s=1.2, c="white")

print(f"Scene: {len(all_objects)} static objects")


## 11. Simulation Loop & Video Export <a id='11'></a>

In [ ]:
import os, imageio
from IPython.display import Video, display

plt = Plotter(
    bg="#04090f", bg2="#080f1a",
    title="Robot-Arm Evacuation Interference",
    offscreen=True, size=(1280, 720)
)

for ri, robot in enumerate(robots):
    robot_actors[ri] = make_robot_visuals(robot)

scene0 = (all_objects
          + robot_actors[0] + robot_actors[1] + robot_actors[2]
          + cluster_markers + predict_markers + [hud])

# Camera: angled to show BOTH floors and the ramp between them
mid_x = (W/2 + FLOOR2_OFFSET_X + W2/2) / 2
plt.show(
    scene0, interactive=False,
    camera=dict(
        pos=(mid_x, -35, 58),
        focal_point=(mid_x, W/2+4, 0),
        viewup=(0, 0, 1)
    )
)

os.makedirs("frames", exist_ok=True)
frames_paths = []
print("Plotter ready.")


In [ ]:
for step in range(steps):
    if not np.any(active):
        print(f"All agents evacuated at step {step}!")
        break

    # 1. Robot IK step
    predicted_targets, cluster_centroids = update_robot_targets(agents, active, robots)

    # 2. Markers
    for ri in range(len(robots)):
        c = cluster_centroids[ri]
        t = predicted_targets[ri]
        if c is not None:
            cluster_markers[ri].pos([c[0], c[1], 0.5]).alpha(0.9)
        if t is not None:
            predict_markers[ri].pos([t[0], t[1], 0.7]).alpha(0.9)

    # 3. Rebuild robot visuals
    for ri, robot in enumerate(robots):
        robot_actors[ri] = make_robot_visuals(robot)

    # 4. Agent dynamics
    new_agents = agents.copy()
    for i, p in enumerate(agents):
        if not active[i]:
            continue
        for e in exits:
            if np.linalg.norm(p[:2] - e[:2]) < exit_tol:
                active[i] = False
                agent_spheres[i].color("#1a2233").alpha(0.12)
                break
        if not active[i]:
            continue

        trails[i].append(p.copy())
        others = agents[np.arange(num_agents) != i]
        g      = gradient(p, prev_agents[i], others)
        move   = -alpha * g
        spd    = np.linalg.norm(move)
        if spd > max_step:
            move = move / spd * max_step
        new_p         = p + move
        new_p[2]      = Z0
        new_agents[i] = new_p
        agent_spheres[i].pos(new_p + [0,0,0.4])

    prev_agents = agents.copy()
    agents      = new_agents

    # 5. Render
    if step % RENDER_EVERY == 0:
        n_evac = int(np.sum(~active))
        pct    = int(100 * n_evac / num_agents)
        hud.text(f"Step: {step:3d} | Evacuated: {n_evac}/{num_agents} ({pct}%)")

        trail_actors = []
        for i in range(num_agents):
            if active[i]:
                tl = list(trails[i])
                nt = len(tl)
                for ti, t_p in enumerate(tl):
                    fade = 0.07 + 0.28 * (ti / max(nt-1, 1))
                    trail_actors.append(
                        Sphere(t_p+[0,0,0.4], r=0.15).color("#ff4422").alpha(fade)
                    )

        plt.clear()
        plt.add(all_objects)
        for ri in range(len(robots)):
            plt.add(robot_actors[ri])
        plt.add(cluster_markers)
        plt.add(predict_markers)
        plt.add(trail_actors)
        plt.add(hud)
        plt.render()

        path = f"frames/frame_{step:04d}.png"
        plt.screenshot(path)
        frames_paths.append(path)

    if step % 50 == 0:
        n_evac = int(np.sum(~active))
        effs = [f"({end_effector_positions[r][0]:.1f},{end_effector_positions[r][1]:.1f})" for r in range(len(robots))]
        print(f"Step {step:3d} | Evac {n_evac}/{num_agents} | Effs: {' '.join(effs)}")

plt.close()
print(f"Done. {len(frames_paths)} frames captured.")


In [ ]:
video_path = "robot_evacuation.mp4"
with imageio.get_writer(
    video_path, fps=VIDEO_FPS, codec="libx264",
    ffmpeg_params=["-pix_fmt", "yuv420p"]
) as writer:
    for fp in frames_paths:
        writer.append_data(imageio.imread(fp))

print(f"Video saved: {video_path}  ({len(frames_paths)} frames @ {VIDEO_FPS} fps)")
display(Video(video_path, embed=True, width=900))


## 12. Summary & Analysis <a id='12'></a>

### Architecture Overview

| Component | Approach |
|-----------|----------|
| Crowd dynamics | Gradient descent on 5-term cost function |
| Robot obstacle | $C_{\text{robot}} = s_{\text{bot}} \cdot \max(0, r_{\text{bot}} - d)^2$ |
| Cluster detection | k-means (k=2) on agents near exits |
| Velocity prediction | Mean displacement × PREDICT_HORIZON steps |
| Robot control | Jacobian pseudo-inverse IK, 1 step per sim step |

### Visualization Legend
| Symbol | Meaning |
|--------|---------|
| 🔴 Red sphere | Active agent |
| ⬛ Dark sphere | Evacuated agent |
| 🟢 Bright green portal | Exit |
| 🟡 Yellow sphere (small) | Detected cluster centroid |
| 🩵 Cyan sphere | Predicted cluster position |
| 🟡 Yellow sphere (large) | Robot end-effector obstacle |
| 🔵 Blue arrow chain | Robot arm segments |
| 🩵 Wireframe bubble | End-effector repulsion radius |

### Experiment Questions
1. Does the robot successfully delay evacuation? Compare with `robot_repulsion_strength = 0`.
2. How does `PREDICT_HORIZON` affect intercept accuracy?
3. What happens with `num_agents = 50`?
4. Try `K_CLUSTERS = 1` — does the arm stop oscillating between targets?
